**Mount Google Drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


**Warning**

In [3]:
import warnings

# Ignore warnings
warnings.filterwarnings('ignore')

**Install Packages**

In [4]:
!pip install pyreadstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 30.6 MB/s eta 0:00:00


In [ ]:
!pip install tensorflow

**Import Libraries**

In [5]:
import numpy as np
import pandas as pd
import pyreadstat
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

**Define Random Seed**

In [6]:
# The random seed
random_seed = 42

# Set random seed in tensorflow
tf.random.set_seed(random_seed)

# Set random seed in numpy
np.random.seed(random_seed)

**Convert SAS File to csv and read the csv file**

In [ ]:
# path to your SAS file
#path = "/content/drive/MyDrive/Colab Notebooks/Project/Dataset/LLCP2024.XPT"

# read SAS file
#df = pd.read_sas(file_path, format="xport", encoding="latin1")

# save as CSV
#df.to_csv("LLCP2024.csv", index=False)



**Set the dataset path**

In [7]:
#read directly from SAS file
#file_path = "/content/drive/MyDrive/Colab Notebooks/teaching/gwu/machine_learning_I/Project/Dataset/LLCP2024.XPT"
#df = pd.read_sas(file_path, format="xport", encoding="latin1")

#read from converted file to csv
file_path="/content/drive/MyDrive/Colab Notebooks/teaching/gwu/machine_learning_I/Project/Dataset/LLCP2024.csv"
df=pd.read_csv(file_path)

**Columns To keep**

In [8]:


cols = [
    'ASTHNOW',
    'SEXVAR',
    'GENHLTH',
    'PHYSHLTH',
    'MENTHLTH',
    'EXERANY2',
    'SMOKE100',
    'ECIGNOW3',
    'CHCCOPD3',
    'DIABETE4',
    'EDUCA',
    'INCOME3',
    'MARITAL',
    '_BMI5',
    '_AGEG5YR',
    'PRIMINS2',
    'MEDCOST1'
]

df2 = df[cols].copy()


**quick exploration on raw selected data**

In [ ]:
print("Raw selected dataset shape:")
print(df2.shape)

print("\nAsthma column before cleaning:")
print(df2['ASTHNOW'].value_counts(dropna=False))

print("\nFirst 5 rows of selected data:")
print(df2.head())

print("\nData types:")
print(df2.dtypes)

 Clean target:
*   keep only valid asthma labels
*   1 = yes, 2 = no

In [9]:
print("Feature(Asthma) column before cleaning:\n")
print(df2['ASTHNOW'].value_counts(dropna=False))

# keep only valid classes (removes NaN, 7, 9 automatically)
final_df = df2[df2['ASTHNOW'].isin([1.0, 2.0])].copy()
final_df['ASTHNOW'] = final_df['ASTHNOW'].map({1.0: 1, 2.0: 0})
final_df = final_df.reset_index(drop=True)
print("Feature(Asthma) column after remove null values,unknown and refused values:\n")
print(final_df['ASTHNOW'].value_counts(dropna=False))


Feature(Asthma) column before cleaning:

ASTHNOW
NaN    385633
1.0     49694
2.0     20267
7.0      2019
9.0        57
Name: count, dtype: int64
Feature(Asthma) column after remove null values,unknown and refused values:

ASTHNOW
1    49694
0    20267
Name: count, dtype: int64


**-Explore the dataset (EDA)**

In [10]:
print("\nDataset size:\n")
print(final_df.shape)
print("\nDataset Information:\n")
print(final_df.info())
print("\n The first 5 rows of dataset:\n")
print(final_df.head())
print("Data set columns names:",final_df.columns)


Dataset size:

(69961, 17)

Dataset Information:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69961 entries, 0 to 69960
Data columns (total 17 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   ASTHNOW   69961 non-null  int64  
 1   SEXVAR    69961 non-null  float64
 2   GENHLTH   69960 non-null  float64
 3   PHYSHLTH  69960 non-null  float64
 4   MENTHLTH  69960 non-null  float64
 5   EXERANY2  69961 non-null  float64
 6   SMOKE100  66109 non-null  float64
 7   ECIGNOW3  65847 non-null  float64
 8   CHCCOPD3  69961 non-null  float64
 9   DIABETE4  69961 non-null  float64
 10  EDUCA     69961 non-null  float64
 11  INCOME3   68750 non-null  float64
 12  MARITAL   69961 non-null  float64
 13  _BMI5     63744 non-null  float64
 14  _AGEG5YR  69961 non-null  float64
 15  PRIMINS2  69961 non-null  float64
 16  MEDCOST1  69960 non-null  float64
dtypes: float64(16), int64(1)
memory usage: 9.1 MB
None

 The first 5 rows of dataset:

   ASTHNOW 

**Fix invalid codes**

In [12]:

final_df['PHYSHLTH'] = final_df['PHYSHLTH'].replace([77, 88, 99], -1)
final_df['MENTHLTH'] = final_df['MENTHLTH'].replace([77, 88, 99], -1)

final_df['INCOME3']  = final_df['INCOME3'].replace([77, 99], -1)

final_df['GENHLTH']  = final_df['GENHLTH'].replace([7, 9], -1)
final_df['SMOKE100'] = final_df['SMOKE100'].replace([7, 9], -1)
final_df['ECIGNOW3'] = final_df['ECIGNOW3'].replace([7, 9], -1)
final_df['MEDCOST1'] = final_df['MEDCOST1'].replace([7, 9], -1)
final_df['PRIMINS2'] = final_df['PRIMINS2'].replace([7, 9], -1)

final_df['EXERANY2'] = final_df['EXERANY2'].replace([7, 9], -1)
final_df['CHCCOPD3'] = final_df['CHCCOPD3'].replace([7, 9], -1)
final_df['DIABETE4'] = final_df['DIABETE4'].replace([7, 9], -1)
final_df['MARITAL']  = final_df['MARITAL'].replace([9], -1)

final_df['_BMI5'] = final_df['_BMI5'] / 100


**Check missing values**

In [11]:
#final_df.isnull().sum()
#missing = final_df.isnull().sum()
print("\nmissing values for each columns:\n")
#missing[missing > 0]
missing = final_df.isnull().sum()
print(missing)


missing values for each columns:

ASTHNOW        0
SEXVAR         0
GENHLTH        1
PHYSHLTH       1
MENTHLTH       1
EXERANY2       0
SMOKE100    3852
ECIGNOW3    4114
CHCCOPD3       0
DIABETE4       0
EDUCA          0
INCOME3     1211
MARITAL        0
_BMI5       6217
_AGEG5YR       0
PRIMINS2       0
MEDCOST1       1
dtype: int64


**Fill null values:**

*  Categorical features: fill with mode
*   Numerical features: fill with median

In [13]:
categorical_cols = ['GENHLTH','SMOKE100','ECIGNOW3','INCOME3','MEDCOST1']

numeric_cols = ['PHYSHLTH','MENTHLTH','_BMI5']


# fill categorical with mode
for col in categorical_cols:
    final_df[col] = final_df[col].fillna(final_df[col].mode()[0])

# fill numeric with median
for col in numeric_cols:
    final_df[col] = final_df[col].fillna(final_df[col].median())

print("Dataset after check missing values:")
print(final_df.isnull().sum())

Dataset after check missing values:
ASTHNOW     0
SEXVAR      0
GENHLTH     0
PHYSHLTH    0
MENTHLTH    0
EXERANY2    0
SMOKE100    0
ECIGNOW3    0
CHCCOPD3    0
DIABETE4    0
EDUCA       0
INCOME3     0
MARITAL     0
_BMI5       0
_AGEG5YR    0
PRIMINS2    0
MEDCOST1    0
dtype: int64


**Class balance check**

In [14]:
print("\nClass distribution in target:\n")
print(final_df['ASTHNOW'].value_counts())

print("\nClass distribution in target (proportion):\n")
print(final_df['ASTHNOW'].value_counts(normalize=True))



Class distribution in target:

ASTHNOW
1    49694
0    20267
Name: count, dtype: int64

Class distribution in target (proportion):

ASTHNOW
1    0.71031
0    0.28969
Name: proportion, dtype: float64


**Split the data for Train and Test**

In [15]:
X = final_df.drop('ASTHNOW', axis=1) #DataSet without the Target
y = final_df['ASTHNOW'] #Target

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

**Test and Train information**

In [16]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

Train shape: (55968, 16)
Test shape: (13993, 16)

Train target distribution:
ASTHNOW
1    0.710317
0    0.289683
Name: proportion, dtype: float64

Test target distribution:
ASTHNOW
1    0.710284
0    0.289716
Name: proportion, dtype: float64


**Scale the data**

In [17]:
from sklearn.preprocessing import StandardScaler

# scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)